# Chapter 12: Optimization Algorithms

This notebook provides implementations and comparisons of the key optimization algorithms used in deep learning:

1. SGD, Momentum, RMSprop, Adam Implementations
2. Side-by-Side Comparison on Test Functions
3. Visualization of Optimizer Trajectories

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
import ipywidgets as widgets
from ipywidgets import interact

# Configure matplotlib
plt.rcParams['figure.figsize'] = [12, 8]
plt.rcParams['font.size'] = 12
np.random.seed(42)

## 1. Optimizer Implementations

Let's implement the main optimization algorithms used in deep learning.

In [ ]:
class SGD:
    """
    Stochastic Gradient Descent with optional momentum.
    
    Update rule:
        v = beta * v + grad
        x = x - lr * v
    """
    def __init__(self, lr=0.01, momentum=0.0):
        self.lr = lr
        self.momentum = momentum
        self.v = None
    
    def step(self, x, grad):
        if self.v is None:
            self.v = np.zeros_like(x)
        
        self.v = self.momentum * self.v + grad
        return x - self.lr * self.v
    
    def reset(self):
        self.v = None

print("SGD: Simple but effective")
print("- With momentum=0: Vanilla gradient descent")
print("- With momentum>0: Accelerated convergence, damped oscillations")

In [ ]:
class RMSprop:
    """
    RMSprop optimizer.
    
    Adapts learning rate based on running average of squared gradients.
    
    Update rule:
        s = beta * s + (1 - beta) * grad^2
        x = x - lr * grad / (sqrt(s) + epsilon)
    """
    def __init__(self, lr=0.01, beta=0.9, epsilon=1e-8):
        self.lr = lr
        self.beta = beta
        self.epsilon = epsilon
        self.s = None
    
    def step(self, x, grad):
        if self.s is None:
            self.s = np.zeros_like(x)
        
        # Update running average of squared gradients
        self.s = self.beta * self.s + (1 - self.beta) * grad**2
        
        # Adaptive update
        return x - self.lr * grad / (np.sqrt(self.s) + self.epsilon)
    
    def reset(self):
        self.s = None

print("RMSprop: Adaptive learning rates")
print("- Divides by running average of gradient magnitudes")
print("- Parameters with large gradients get smaller effective learning rates")
print("- Good for non-stationary objectives")

In [ ]:
class Adam:
    """
    Adam (Adaptive Moment Estimation) optimizer.
    
    Combines momentum with adaptive learning rates.
    
    Update rule:
        m = beta1 * m + (1 - beta1) * grad           # First moment
        v = beta2 * v + (1 - beta2) * grad^2         # Second moment
        m_hat = m / (1 - beta1^t)                    # Bias correction
        v_hat = v / (1 - beta2^t)                    # Bias correction
        x = x - lr * m_hat / (sqrt(v_hat) + epsilon)
    """
    def __init__(self, lr=0.001, beta1=0.9, beta2=0.999, epsilon=1e-8):
        self.lr = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.epsilon = epsilon
        self.m = None
        self.v = None
        self.t = 0
    
    def step(self, x, grad):
        if self.m is None:
            self.m = np.zeros_like(x)
            self.v = np.zeros_like(x)
        
        self.t += 1
        
        # Update biased first and second moment estimates
        self.m = self.beta1 * self.m + (1 - self.beta1) * grad
        self.v = self.beta2 * self.v + (1 - self.beta2) * grad**2
        
        # Bias correction
        m_hat = self.m / (1 - self.beta1**self.t)
        v_hat = self.v / (1 - self.beta2**self.t)
        
        return x - self.lr * m_hat / (np.sqrt(v_hat) + self.epsilon)
    
    def reset(self):
        self.m = None
        self.v = None
        self.t = 0

print("Adam: The most popular optimizer")
print("- Combines momentum (first moment) with RMSprop (second moment)")
print("- Bias correction for better early-training behavior")
print("- Works well with default hyperparameters")

In [ ]:
class AdaGrad:
    """
    AdaGrad optimizer.
    
    Accumulates squared gradients over all time.
    Good for sparse gradients, but learning rate decays to zero.
    
    Update rule:
        s = s + grad^2
        x = x - lr * grad / (sqrt(s) + epsilon)
    """
    def __init__(self, lr=0.01, epsilon=1e-8):
        self.lr = lr
        self.epsilon = epsilon
        self.s = None
    
    def step(self, x, grad):
        if self.s is None:
            self.s = np.zeros_like(x)
        
        self.s = self.s + grad**2
        return x - self.lr * grad / (np.sqrt(self.s) + self.epsilon)
    
    def reset(self):
        self.s = None

print("AdaGrad: Adaptive learning with accumulation")
print("- Learning rate decreases over time (can become too small)")
print("- Good for sparse features")

In [ ]:
def optimize(optimizer, grad_fn, x_init, n_steps=100):
    """
    Run optimization and return the path.
    
    Parameters:
        optimizer: Optimizer instance (SGD, RMSprop, Adam, etc.)
        grad_fn: Function that computes gradient at a point
        x_init: Initial parameter values
        n_steps: Number of optimization steps
    
    Returns:
        path: List of parameter values at each step
    """
    optimizer.reset()
    x = np.array(x_init, dtype=float)
    path = [x.copy()]
    
    for _ in range(n_steps):
        grad = grad_fn(x)
        x = optimizer.step(x, grad)
        path.append(x.copy())
    
    return path

## 2. Side-by-Side Comparison on Test Functions

Let's compare all optimizers on various test functions.

In [ ]:
# Define test functions and their gradients
class TestFunction:
    """Container for a test function and its gradient."""
    def __init__(self, f, grad, name, x_init, xlim, ylim, minimum):
        self.f = f
        self.grad = grad
        self.name = name
        self.x_init = x_init
        self.xlim = xlim
        self.ylim = ylim
        self.minimum = minimum

# Quadratic bowl
quadratic = TestFunction(
    f=lambda x: x[0]**2 + 2*x[1]**2,
    grad=lambda x: np.array([2*x[0], 4*x[1]]),
    name='Quadratic',
    x_init=np.array([4.0, 3.0]),
    xlim=(-5, 5),
    ylim=(-4, 4),
    minimum=np.array([0.0, 0.0])
)

# Elongated quadratic (ill-conditioned)
elongated = TestFunction(
    f=lambda x: 0.1*x[0]**2 + 10*x[1]**2,
    grad=lambda x: np.array([0.2*x[0], 20*x[1]]),
    name='Elongated Quadratic',
    x_init=np.array([10.0, 1.0]),
    xlim=(-12, 12),
    ylim=(-2, 2),
    minimum=np.array([0.0, 0.0])
)

# Rosenbrock (banana function)
rosenbrock = TestFunction(
    f=lambda x: (1 - x[0])**2 + 100*(x[1] - x[0]**2)**2,
    grad=lambda x: np.array([
        -2*(1 - x[0]) - 400*x[0]*(x[1] - x[0]**2),
        200*(x[1] - x[0]**2)
    ]),
    name='Rosenbrock',
    x_init=np.array([-1.5, 2.0]),
    xlim=(-2, 2),
    ylim=(-1, 3),
    minimum=np.array([1.0, 1.0])
)

# Beale function
def beale_f(x):
    return ((1.5 - x[0] + x[0]*x[1])**2 + 
            (2.25 - x[0] + x[0]*x[1]**2)**2 + 
            (2.625 - x[0] + x[0]*x[1]**3)**2)

def beale_grad(x):
    t1 = 1.5 - x[0] + x[0]*x[1]
    t2 = 2.25 - x[0] + x[0]*x[1]**2
    t3 = 2.625 - x[0] + x[0]*x[1]**3
    
    dx0 = 2*t1*(-1 + x[1]) + 2*t2*(-1 + x[1]**2) + 2*t3*(-1 + x[1]**3)
    dx1 = 2*t1*x[0] + 2*t2*(2*x[0]*x[1]) + 2*t3*(3*x[0]*x[1]**2)
    return np.array([dx0, dx1])

beale = TestFunction(
    f=beale_f,
    grad=beale_grad,
    name='Beale',
    x_init=np.array([0.0, 0.0]),
    xlim=(-4, 4),
    ylim=(-4, 4),
    minimum=np.array([3.0, 0.5])
)

test_functions = [quadratic, elongated, rosenbrock, beale]

In [ ]:
def compare_optimizers_on_function(test_func, n_steps=100):
    """
    Compare all optimizers on a given test function.
    """
    # Create optimizers with reasonable default learning rates
    optimizers = {
        'SGD': SGD(lr=0.05),
        'SGD+Momentum': SGD(lr=0.05, momentum=0.9),
        'RMSprop': RMSprop(lr=0.05),
        'Adam': Adam(lr=0.1),
    }
    
    colors = {'SGD': 'blue', 'SGD+Momentum': 'orange', 'RMSprop': 'green', 'Adam': 'red'}
    
    # Run optimization
    paths = {}
    for name, opt in optimizers.items():
        paths[name] = optimize(opt, test_func.grad, test_func.x_init, n_steps)
    
    # Visualize
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Contour plot
    ax = axes[0]
    x_range = np.linspace(test_func.xlim[0], test_func.xlim[1], 100)
    y_range = np.linspace(test_func.ylim[0], test_func.ylim[1], 100)
    X, Y = np.meshgrid(x_range, y_range)
    Z = np.zeros_like(X)
    for i in range(X.shape[0]):
        for j in range(X.shape[1]):
            Z[i, j] = test_func.f(np.array([X[i, j], Y[i, j]]))
    
    # Use log scale for contours if range is large
    if Z.max() / max(Z.min(), 1e-10) > 100:
        levels = np.logspace(np.log10(max(Z.min(), 1e-10)), np.log10(Z.max()), 30)
    else:
        levels = 30
    
    ax.contour(X, Y, Z, levels=levels, cmap='viridis', alpha=0.5)
    ax.contourf(X, Y, Z, levels=levels, cmap='viridis', alpha=0.2)
    
    # Plot paths
    for name, path in paths.items():
        path_arr = np.array(path)
        ax.plot(path_arr[:, 0], path_arr[:, 1], '.-', color=colors[name],
                markersize=3, linewidth=1.5, label=name, alpha=0.8)
    
    # Mark start and minimum
    ax.scatter(test_func.x_init[0], test_func.x_init[1], color='black', 
               s=100, marker='o', zorder=5, label='Start')
    ax.scatter(test_func.minimum[0], test_func.minimum[1], color='gold', 
               s=150, marker='*', zorder=5, edgecolors='black', label='Minimum')
    
    ax.set_xlabel('$x_1$')
    ax.set_ylabel('$x_2$')
    ax.set_title(f'{test_func.name} Function: Optimizer Trajectories')
    ax.legend(loc='upper right')
    
    # Convergence plot
    ax = axes[1]
    for name, path in paths.items():
        losses = [test_func.f(p) for p in path]
        ax.semilogy(losses, color=colors[name], linewidth=2, label=name)
    
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Loss (log scale)')
    ax.set_title('Convergence Comparison')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Print final losses
    print(f"\n{test_func.name} - Final Losses after {n_steps} steps:")
    print("-" * 40)
    for name, path in paths.items():
        final_loss = test_func.f(path[-1])
        distance = np.linalg.norm(path[-1] - test_func.minimum)
        print(f"{name:15s}: loss = {final_loss:.6e}, dist to min = {distance:.4f}")

In [ ]:
# Compare on quadratic
compare_optimizers_on_function(quadratic, n_steps=50)

In [ ]:
# Compare on elongated quadratic (tests handling of ill-conditioning)
compare_optimizers_on_function(elongated, n_steps=100)

In [ ]:
# Compare on Rosenbrock (challenging non-convex landscape)
compare_optimizers_on_function(rosenbrock, n_steps=1000)

In [ ]:
# Compare on Beale function
compare_optimizers_on_function(beale, n_steps=500)

## 3. Detailed Visualization of Optimizer Behaviors

Let's examine specific behaviors of each optimizer.

In [ ]:
# Demonstrate Adam's bias correction
class AdamNoBiasCorrection:
    """Adam without bias correction for comparison."""
    def __init__(self, lr=0.001, beta1=0.9, beta2=0.999, epsilon=1e-8):
        self.lr = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.epsilon = epsilon
        self.m = None
        self.v = None
    
    def step(self, x, grad):
        if self.m is None:
            self.m = np.zeros_like(x)
            self.v = np.zeros_like(x)
        
        self.m = self.beta1 * self.m + (1 - self.beta1) * grad
        self.v = self.beta2 * self.v + (1 - self.beta2) * grad**2
        
        # No bias correction!
        return x - self.lr * self.m / (np.sqrt(self.v) + self.epsilon)
    
    def reset(self):
        self.m = None
        self.v = None

# Compare Adam with and without bias correction
adam_with = Adam(lr=0.3)
adam_without = AdamNoBiasCorrection(lr=0.3)

path_with = optimize(adam_with, quadratic.grad, quadratic.x_init, n_steps=50)
path_without = optimize(adam_without, quadratic.grad, quadratic.x_init, n_steps=50)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Early iterations focus
ax = axes[0]
losses_with = [quadratic.f(p) for p in path_with]
losses_without = [quadratic.f(p) for p in path_without]

ax.semilogy(losses_with[:20], 'b-', linewidth=2, label='Adam (with bias correction)')
ax.semilogy(losses_without[:20], 'r--', linewidth=2, label='Adam (no bias correction)')
ax.set_xlabel('Iteration')
ax.set_ylabel('Loss')
ax.set_title('Early Training: Bias Correction Effect')
ax.legend()
ax.grid(True, alpha=0.3)

# Full trajectory
ax = axes[1]
x_range = np.linspace(-5, 5, 100)
y_range = np.linspace(-4, 4, 100)
X, Y = np.meshgrid(x_range, y_range)
Z = X**2 + 2*Y**2

ax.contour(X, Y, Z, levels=20, cmap='viridis', alpha=0.5)

path_w = np.array(path_with[:20])
path_wo = np.array(path_without[:20])

ax.plot(path_w[:, 0], path_w[:, 1], 'b.-', markersize=6, linewidth=2, label='With bias correction')
ax.plot(path_wo[:, 0], path_wo[:, 1], 'r.--', markersize=6, linewidth=2, label='No bias correction')

ax.set_xlabel('$x_1$')
ax.set_ylabel('$x_2$')
ax.set_title('Early Training Trajectories')
ax.legend()

plt.tight_layout()
plt.show()

print("Bias correction compensates for the initialization of m and v at zero.")
print("Without it, early steps are too small because m and v haven't accumulated enough information.")

In [ ]:
# Visualize adaptive learning rates
def track_effective_lr(optimizer_class, optimizer_kwargs, grad_fn, x_init, n_steps=50):
    """
    Track the effective learning rate for each parameter dimension.
    """
    opt = optimizer_class(**optimizer_kwargs)
    x = np.array(x_init, dtype=float)
    
    effective_lrs = []
    path = [x.copy()]
    
    for _ in range(n_steps):
        grad = grad_fn(x)
        x_old = x.copy()
        x = opt.step(x, grad)
        
        # Effective learning rate = |delta x| / |grad|
        delta = x - x_old
        eff_lr = np.abs(delta) / (np.abs(grad) + 1e-10)
        effective_lrs.append(eff_lr)
        path.append(x.copy())
    
    return np.array(effective_lrs), path

# Compare effective learning rates
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

optimizers_info = [
    ('SGD', SGD, {'lr': 0.1}),
    ('RMSprop', RMSprop, {'lr': 0.1}),
    ('Adam', Adam, {'lr': 0.3}),
    ('AdaGrad', AdaGrad, {'lr': 0.5}),
]

for ax, (name, opt_class, opt_kwargs) in zip(axes.flatten(), optimizers_info):
    eff_lrs, _ = track_effective_lr(opt_class, opt_kwargs, elongated.grad, elongated.x_init, n_steps=100)
    
    ax.semilogy(eff_lrs[:, 0], 'b-', linewidth=2, label='$x_1$ (shallow direction)')
    ax.semilogy(eff_lrs[:, 1], 'r-', linewidth=2, label='$x_2$ (steep direction)')
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Effective Learning Rate')
    ax.set_title(f'{name}')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('Effective Learning Rates on Elongated Quadratic', fontsize=14)
plt.tight_layout()
plt.show()

print("Key observations:")
print("- SGD: Same effective LR for both dimensions (problematic for ill-conditioned problems)")
print("- RMSprop/Adam: Adapts LR per dimension (smaller for steep, larger for shallow)")
print("- AdaGrad: Effective LR decreases over time (can become too small)")

In [ ]:
# Interactive optimizer comparison
def interactive_optimizer_comparison(function='Quadratic', optimizer='Adam', lr=0.1, 
                                      momentum=0.9, n_steps=100):
    """
    Interactively explore optimizer behavior.
    """
    # Select function
    func_map = {
        'Quadratic': quadratic,
        'Elongated': elongated,
        'Rosenbrock': rosenbrock,
        'Beale': beale
    }
    test_func = func_map[function]
    
    # Create optimizer
    if optimizer == 'SGD':
        opt = SGD(lr=lr)
    elif optimizer == 'SGD+Momentum':
        opt = SGD(lr=lr, momentum=momentum)
    elif optimizer == 'RMSprop':
        opt = RMSprop(lr=lr)
    elif optimizer == 'Adam':
        opt = Adam(lr=lr)
    else:
        opt = AdaGrad(lr=lr)
    
    # Run optimization
    path = optimize(opt, test_func.grad, test_func.x_init, n_steps)
    
    # Visualize
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Contour
    ax = axes[0]
    x_range = np.linspace(test_func.xlim[0], test_func.xlim[1], 100)
    y_range = np.linspace(test_func.ylim[0], test_func.ylim[1], 100)
    X, Y = np.meshgrid(x_range, y_range)
    Z = np.zeros_like(X)
    for i in range(X.shape[0]):
        for j in range(X.shape[1]):
            Z[i, j] = test_func.f(np.array([X[i, j], Y[i, j]]))
    
    ax.contour(X, Y, Z, levels=30, cmap='viridis', alpha=0.5)
    ax.contourf(X, Y, Z, levels=30, cmap='viridis', alpha=0.2)
    
    path_arr = np.array(path)
    ax.plot(path_arr[:, 0], path_arr[:, 1], 'r.-', markersize=4, linewidth=1.5)
    ax.scatter(path_arr[0, 0], path_arr[0, 1], color='red', s=100, zorder=5)
    ax.scatter(test_func.minimum[0], test_func.minimum[1], color='gold', 
               s=150, marker='*', zorder=5, edgecolors='black')
    
    ax.set_xlabel('$x_1$')
    ax.set_ylabel('$x_2$')
    ax.set_title(f'{function} - {optimizer}')
    
    # Loss
    ax = axes[1]
    losses = [test_func.f(p) for p in path]
    ax.semilogy(losses, 'b-', linewidth=2)
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Loss (log scale)')
    ax.set_title('Convergence')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f"Final loss: {losses[-1]:.6e}")
    print(f"Final position: {path[-1]}")
    print(f"Distance to minimum: {np.linalg.norm(path[-1] - test_func.minimum):.4f}")

interact(interactive_optimizer_comparison,
         function=['Quadratic', 'Elongated', 'Rosenbrock', 'Beale'],
         optimizer=['SGD', 'SGD+Momentum', 'RMSprop', 'Adam', 'AdaGrad'],
         lr=widgets.FloatSlider(min=0.001, max=1.0, step=0.01, value=0.1, description='Learning Rate'),
         momentum=widgets.FloatSlider(min=0, max=0.99, step=0.05, value=0.9, description='Momentum'),
         n_steps=widgets.IntSlider(min=10, max=1000, step=10, value=100, description='Steps'));

In [ ]:
# Summary comparison table
print("="*80)
print("OPTIMIZER COMPARISON SUMMARY")
print("="*80)
print()

summary = """
| Optimizer     | Key Idea                      | Pros                           | Cons                          |
|---------------|-------------------------------|--------------------------------|-------------------------------|
| SGD           | x = x - lr * grad             | Simple, generalizes well       | Slow on ill-conditioned       |
| SGD+Momentum  | Accumulate past gradients     | Faster, dampens oscillations   | Extra hyperparameter          |
| AdaGrad       | Accumulate squared gradients  | Good for sparse features       | LR decays to zero             |
| RMSprop       | Decaying average of grad^2   | Adaptive LR, non-stationary    | Not as robust as Adam         |
| Adam          | Momentum + RMSprop + bias     | Works well by default          | May not generalize as well    |
"""
print(summary)

print("\nPractical Guidelines:")
print("-" * 40)
print("1. Start with Adam (lr=0.001) - works well by default")
print("2. For best generalization, try SGD+Momentum")
print("3. Tune learning rate first (most important hyperparameter)")
print("4. Use learning rate scheduling for better convergence")

## Summary

In this notebook, we implemented and compared the key optimization algorithms:

1. **SGD**: Simple gradient descent, optionally with momentum
2. **RMSprop**: Adaptive learning rates via running average of squared gradients
3. **Adam**: Combines momentum with adaptive learning rates, plus bias correction
4. **AdaGrad**: Accumulates squared gradients (learning rate decays over time)

### Key Takeaways:

- **Adam** is a good default choice - works well with default hyperparameters
- **SGD+Momentum** may generalize better but requires more tuning
- **Adaptive methods** (RMSprop, Adam) handle ill-conditioned problems better
- **Learning rate** is the most important hyperparameter to tune

### When to Use What:

- **Adam**: General-purpose, especially good for training transformers and LLMs
- **SGD+Momentum**: Computer vision (often with learning rate schedule)
- **RMSprop**: Recurrent neural networks
- **AdaGrad**: Sparse features (NLP with embeddings)

---

*This concludes the optimization algorithms chapter.*